# Phase 13 Part 1A — Historical Localizer Reconstruction + CLEAN-v3 Protocol Freeze

**Date:** 2026-09-21
**Manifest:** `phase4_v3` · **Taxonomy:** `kub_size_clean_v3_v1`
**Strategy revision:** SR-1 (Regional-Aware Multi-Domain FiLM) · final system `RA-MedFiLM-Seg++` (`PROPOSED_NOT_IMPLEMENTED`)

---

> **PROTOCOL / DEFINITION PHASE ONLY.** Part 1A reconstructs the historical Phase-13 /
> Phase-13B localizer protocol and prepares a **prospective** clean-v3 localizer + proposal
> protocol candidate.
>
> **No localizer training. No localizer inference. No proposal generation.**
> No checkpoint/history/config/manifest changes. No TEST access. Phase 14 not executed.
> Phases 14–18 not started. RA-MedFiLM not implemented. No winner or single reference
> architecture selected. `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13` remains **NOT_DECIDED**.
>
> **Primary scientific question (Phase 13):** *can a GT-free region proposal / localization
> mechanism retain enough of the Phase-12 oracle regional-resolution benefit while
> controlling FP burden and preserving global segmentation quality?*


## 1. Scope and Governance

| Rule | Status |
|---|---|
| Localizer training epochs / optimizer steps | **0 / 0** |
| Localizer inference passes | **0** |
| Proposals generated on real images | **0** |
| Checkpoint / history / config / manifest changes | **0** |
| TEST image/mask/content reads | **0** |
| Phase 14 executed | **NO** |
| Phases 15–18 | **NOT STARTED** |
| RA-MedFiLM implementation | **0** |
| Winner / single reference architecture | **NONE** |
| `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13` | **NOT_DECIDED** |


In [1]:
import sys, json
from pathlib import Path
import pandas as pd
pd.set_option('display.max_rows', None); pd.set_option('display.max_columns', None)
pd.set_option('display.width', None); pd.set_option('display.max_colwidth', 80)

PROJECT_ROOT = Path(r'D:\Softfors\jup')
CODE_DIR = PROJECT_ROOT / 'notebooks/phasewise/phase_13_predicted_roi_localization/code'
for p in (str(PROJECT_ROOT), str(CODE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

import phase13_localizer_protocol as p13
TABLES = PROJECT_ROOT / 'reports/tables'
AUDITS = PROJECT_ROOT / 'reports/audits'

# Read-only: regenerate the tables from the frozen artifacts so the notebook and the
# module provably produce identical output. Nothing is trained, inferred or generated.
hashes = p13.write_all(write_audits=True)
print(f'wrote {len(hashes)} artifacts')
print(f'TEST content reads: 0  |  training: 0  |  inference: 0  |  proposals: 0')


wrote 19 artifacts
TEST content reads: 0  |  training: 0  |  inference: 0  |  proposals: 0


## 2. Historical Population (phase4_v2)

The historical Phase-13 / Phase-13B localizer study ran under **`phase4_v2`**, not clean-v3.
This is the single most important reconstruction fact: the frozen Stage-4D numbers
(Tiny 53/73, overall 100/156) belong to the **historical** population and must not be
compared with clean-v3 numbers as if the splits matched.


In [2]:
hist_pop = pd.DataFrame([
    {'split': 'TRAIN', 'acquisition': 'FULL_FIELD', 'images': 771, 'components': 1162},
    {'split': 'VAL', 'acquisition': 'FULL_FIELD', 'images': 106, 'components': 156},
    {'split': 'VAL', 'acquisition': 'ROI_CROP', 'images': 177, 'components': 333 - 156},
])
display(hist_pop)
print('historical VAL FULL_FIELD size strata (kub_size_v1):', p13.VAL_STRATA_HIST)
print('historical manifest:', p13.MANIFEST_HIST, '| taxonomy:', p13.TAXONOMY_HIST)


,split,acquisition,images,components
0,TRAIN,FULL_FIELD,771,1162
1,VAL,FULL_FIELD,106,156
2,VAL,ROI_CROP,177,177


historical VAL FULL_FIELD size strata (kub_size_v1): {'TINY': 73, 'SMALL': 29, 'MEDIUM': 36, 'LARGE': 18}
historical manifest: phase4_v2 | taxonomy: kub_size_v1


## 3. Historical Source Inventory

Reconstructed with the source-precedence ladder: executing source (rank 1) > persisted
audit/table (rank 2) > executed notebook (rank 3) > frozen protocol/README (rank 4) >
prose. Downstream Phase-14 audits are included at rank 2 as **evidence of the historical
train/inference relationship**, not as a Phase-13 source of truth.


In [3]:
src_inv = pd.read_csv(TABLES / 'phase13_historical_source_inventory.csv')
display(src_inv[['source_path', 'source_type', 'authority_rank', 'notes']])
print('MISSING sources:', src_inv[src_inv['notes'] != 'present']['source_path'].tolist() or 'none')


,source_path,source_type,authority_rank,notes
0,src/models/kub_localizer.py,executing_source,1,present
1,src/inference/learned_roi.py,executing_source,1,present
2,src/inference/predicted_roi.py,executing_source,1,present
3,src/inference/heatmap_targets.py,executing_source,1,present
4,src/data/kub_localizer_dataset.py,executing_source,1,present
5,configs/kub/phase13b_localizer.yaml,frozen_config,2,present
6,PHASE13B.md,frozen_protocol,2,present
7,reports/audits/phase13_predicted_roi_localizer.json,persisted_audit,2,present
8,reports/audits/phase13b_stage3_validation_correction.md,persisted_audit,2,present
9,reports/audits/phase13b_stage4b_feasible_selection.md,persisted_audit,2,present


MISSING sources: none


## 4. Historical Stage Inventory

Phase 13 developed through eight recorded stages, including three failed / corrected
iterations. The final Stage-4D-R2 is the authoritative localizer.


In [4]:
stages = pd.read_csv(TABLES / 'phase13_historical_stage_inventory.csv')
display(stages[['stage_id', 'stage_name', 'experiment_id', 'status']])
display(stages[['stage_id', 'outcome_summary']])


,stage_id,stage_name,experiment_id,status
0,13,Phase-13 SIMPLE predicted-ROI localizer (no learning),phase13_predicted_roi,COMPLETE_PASS_GATE_WEAK
1,13B-4A,Learned localizer pilot v1 (unconstrained recall objective),phase13b_kub_localizer_r512_s3_seed42,INVALID_FOR_FINAL_SELECTION
2,13B-4B,Feasibility-constrained localization,phase13b_kub_localizer_r512_s3_seed42_v3,FAIL_CHECK_REQUIRED
3,13B-4C,CenterNet-style focal loss redesign pilot,phase13b_kub_localizer_r512_s3_centernetfocal_seed42_v1,FAIL_CHECK_REQUIRED
4,13B-4C-R3,Component mapping correction,phase13b_kub_localizer_r512_s3_centernetfocal_seed42_v2,PASS
5,13B-4C-R4,CenterNet focal pilot re-run,phase13b_kub_localizer_r512_s3_centernetfocal_seed42_v3,PILOT_BELOW_GATE
6,13B-4D-R1,Final bounded localizer preflight (stride-4 / 128 output),phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1,PREFLIGHT_PASS
7,13B-4D-R2,Final Stage-4D localizer (authoritative),phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1,COMPLETE_FROZEN


,stage_id,outcome_summary
0,13,overall recall90 0.743590; Tiny recall90 0.684932; 0/16 hard Tiny; 0/7 globa...
1,13B-4A,Degenerate full-image ROIs achieved perfect recall at ROI area fraction 1.0;...
2,13B-4B,"Feasible configs bounded correctly but recall collapsed (Tiny 1.37%, overall..."
3,13B-4C,Target validation FAILED: sub-pixel centres gave peak density < 1.0 so targe...
4,13B-4C-R3,"1162/1162 mapped, 0 legitimately-outside, 1162 unique canonical centres, 0 c..."
5,13B-4C-R4,"Epoch 97: Tiny recall90 0.465753, overall recall90 0.301282, hard Tiny 1/16,..."
6,13B-4D-R1,"Params unchanged 482,449; constant logit init commutes exactly (p=0.100000);..."
7,13B-4D-R2,best_epoch 73 / 88 total; Tiny recall90 53/73 = 0.726027; overall 100/156 = ...


## 5. Historical Localizer Protocol Matrix

Every concept reconstructed from the executing source or a persisted artifact. Note the
selection rule: Phase 13 is **localization-only** — segmentation Dice is deliberately
never a localizer selection or reporting target.


In [5]:
proto = pd.read_csv(TABLES / 'phase13_historical_localizer_protocol_matrix.csv')
display(proto[['concept', 'historical_value', 'unit', 'status']])
print('status counts:', proto['status'].value_counts().to_dict())


,concept,historical_value,unit,status
0,A_model_scope,KUBLocalizer — compact 3-stage U-Net (base_channels 16),model,VERIFIED
1,B_parameter_count,"482,449",params,VERIFIED
2,C_input,"grayscale FULL_FIELD image, 1 channel",tensor,VERIFIED
3,D_output,one-channel objectness logit heatmap,tensor,VERIFIED
4,E_initialization,random Kaiming; no pretrained weights; head bias log(0.1/0.9),rule,VERIFIED
5,F_supervision,TRAIN-only soft centre heatmaps derived from GT component centroids,rule,VERIFIED
6,G_heatmap_target_rule,"Gaussian, pixelwise MAX across components; nearest integer pixel forced to 1.0",rule,VERIFIED
7,H_sigma_grid,"3, 5, 7 (512-space); selected equivalent 0.75 in 128-space",px,VERIFIED
8,I_resolution_grid,"512, 768 (6 candidates, not expanded)",px,VERIFIED
9,J_loss,soft_focal_dice: 0.75 CenterNet focal + 0.25 soft Dice,loss,VERIFIED


status counts: {'VERIFIED': 35, 'VERIFIED_DEFECT': 1, 'AMBIGUOUS': 1}


## 6. Historical Checkpoint Provenance + Handoff-Claim Verification

All nine claims in the Phase-13 handoff note are re-verified directly against the frozen
Stage-4D evaluation artifact and the on-disk checkpoint bytes.


In [6]:
ckpt = pd.read_csv(TABLES / 'phase13_historical_checkpoint_provenance.csv')
display(ckpt[['experiment_id', 'role', 'sha256', 'sha256_matches_claim', 'best_epoch', 'status']])

handoff = pd.read_csv(TABLES / 'phase13_historical_handoff_verification.csv')
display(handoff)
assert (handoff['verdict'] == 'VERIFIED').all(), 'handoff claim mismatch'
print('ALL HANDOFF CLAIMS VERIFIED')


,experiment_id,role,sha256,sha256_matches_claim,best_epoch,status
0,phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1,BEST_SELECTED,ece906df6c73f2e93c3f873661490bf3cb1a6b82be2d2d93b58c25bc967cbace,True,73.0,AUTHORITATIVE
1,phase13b_kub_localizer_r512_h128_sigmaeq075_centernetfocal_seed42_v1,LATEST,a1461df2e8d6784984337c79b87c65833c39fed16382579088d97d7c60ba78f4,True,NaN,RESUME_ONLY
2,phase13b_kub_localizer_r512_s3_centernetfocal_seed42_v3,BEST,260ed9828aed9c19b62a5a58ea9321bbce9aeefc221db4ac3193e37e34eb6449,NaN,97.0,SUPERSEDED
3,phase13b_kub_localizer_r512_s3_centernetfocal_seed42_v1,BEST,7749c892e702b0953a8f8a047b5134674bd60475c8d7aca93ff8ea457ed9b0fc,NaN,NaN,INVALID_FOR_FINAL_SELECTION
4,phase13b_kub_localizer_r512_s3_seed42_v3,BEST,7a188effafbc8dd0b91efc4ee0f41097728579121a927087b868c8f23920e54d,NaN,11.0,FAIL_CHECK_REQUIRED
5,phase13b_kub_localizer_r512_s3_seed42,BEST,7c6226fd6199fbc564f24e27ec4f974cf1eb3e31622fb6546bcf6eea5f6341df,NaN,NaN,INVALID_FOR_FINAL_SELECTION


,claim,claimed,actual,verdict
0,Stage-4D best.pt SHA-256,ece906df6c73f2e93c3f873661490bf3cb1a6b82be2d2d93b58c25bc967cbace,ece906df6c73f2e93c3f873661490bf3cb1a6b82be2d2d93b58c25bc967cbace,VERIFIED
1,Stage-4D best_epoch,73,73,VERIFIED
2,Stage-4D total_epochs,88,88,VERIFIED
3,selected threshold,0.1,0.1,VERIFIED
4,selected margin,0.2,0.2,VERIFIED
5,selected top_k,5,5,VERIFIED
6,selected nms_iou,0.3,0.3,VERIFIED
7,Tiny recall90 numerator/denominator,53/73,53/73,VERIFIED
8,Overall recall90 numerator/denominator,100/156,100/156,VERIFIED


ALL HANDOFF CLAIMS VERIFIED


## 7. Findings

### 7.1 `HISTORICAL_TRAIN_INFERENCE_PROPOSAL_DISTRIBUTION_MISMATCH` — **VERIFIED**

The historical Phase-14 regional refiner was **trained** on GT-derived base crops plus
GT-perturbed crops plus random background negatives (Option A3: 1162 + 2324 + 1542 = 5028
crops/epoch), but **evaluated** on frozen Stage-4D **predicted** proposals (Condition C).

The Phase-14 Step-3 failure (`D_p = 0.383905` vs `D_g = 0.652776`, driven by 5.43 FP
components/image vs 0.37 baseline) is therefore at least partly attributable to this
distribution gap and is **not** a clean measurement of predicted-proposal regional
refinement.


In [7]:
rec = json.loads((AUDITS / 'phase13_historical_localizer_protocol_reconstruction.json').read_text(encoding='utf-8'))
find = rec['findings']
for k, v in find.items():
    print(f"{k}: {v['status']}")
print()
mismatch = find['HISTORICAL_TRAIN_INFERENCE_PROPOSAL_DISTRIBUTION_MISMATCH']
print('EVIDENCE:')
for e in mismatch['evidence']:
    print(' -', e)
print()
print('STATEMENT:', mismatch['statement'])
print()
print('CONSEQUENCE:', mismatch['consequence'])
print()
print('CLEAN-V3 ACTION:', mismatch['clean_v3_action'])


HISTORICAL_TRAIN_INFERENCE_PROPOSAL_DISTRIBUTION_MISMATCH: VERIFIED
LATENT_INFERENCE_HEATMAP_RESOLUTION_OMISSION: VERIFIED_DEFECT
STAGE4D_DESIGN_AUDIT_LABEL_AMBIGUITY: AMBIGUOUS
MISSING_VAL_SEARCH_EPOCH_079: VERIFIED

EVIDENCE:
 - reports/audits/phase14_diagnostic_protocol_amendment.md §6 (A3 policy): 1162 GT-derived base crops + 2324 perturbed crops + 1542 background negatives = 5028 crops/epoch
 - reports/audits/phase14_part1_implementation_preflight.md: dataset composition 5028 (1162 / 2324 / 1542)
 - reports/audits/phase14_diagnostic_protocol_amendment.md §3 Condition C: the same refiner is evaluated on frozen Stage-4D PREDICTED proposals

STATEMENT: The historical Phase-14 regional refiner was TRAINED on GT-derived base crops, GT-perturbed crops and random background negatives, but EVALUATED on Stage-4D predicted proposals. The training and inference proposal distributions therefore did not match.

CONSEQUENCE: The Phase-14 Step-3 failure (D_p 0.383905 vs D_g 0.652776, driven by 5

### 7.2 Latent inference defect, label ambiguity, and an artifact gap

Three further findings were recorded. The latent defect is a **verified** bug in the
historical inference helper that must be fixed (or an output-resolution contract frozen)
before any clean-v3 reuse.


In [8]:
for k in ['LATENT_INFERENCE_HEATMAP_RESOLUTION_OMISSION',
          'STAGE4D_DESIGN_AUDIT_LABEL_AMBIGUITY',
          'MISSING_VAL_SEARCH_EPOCH_079']:
    v = find[k]
    print(f"--- {k} [{v['status']}]")
    print(v['statement'])
    print('   action:', v['clean_v3_action'])
    print()

print('Phase-14 historical gate arithmetic (v2, coverage90):', p13.PHASE14_GATE)
print('Phase-14 classification:', rec['phase14_classification'])


--- LATENT_INFERENCE_HEATMAP_RESOLUTION_OMISSION [VERIFIED_DEFECT]
The Stage-4D checkpoint was trained with a 128x128 output heatmap (AvgPool2d(4,4)). The inference helper would construct the 512-output variant. Because the logit downsample is parameter-free and the head is a 1x1 convolution, the state dict still loads, so the mismatch is silent and the helper would return a 512-space heatmap.
   action: Pass heatmap_resolution explicitly (or freeze an output-resolution contract) and add a regression test.

--- STAGE4D_DESIGN_AUDIT_LABEL_AMBIGUITY [AMBIGUOUS]
The label most plausibly refers to the clean-v3 ARCHITECTURE (KUBLocalizer), not the clean-v3 MANIFEST.
   action: Supervisor confirmation before any reuse of the Stage-4C numbers.

--- MISSING_VAL_SEARCH_EPOCH_079 [VERIFIED]
Only the per-epoch proposal-grid dump for epoch 079 is missing; the epoch history itself is complete.
   action: None required for Phase 13; noted for provenance.

Phase-14 historical gate arithmetic (v2, cov

## 8. Clean-v3 Population Specification

The clean-v3 FULL_FIELD population was recomputed from the manifest and a TRAIN/VAL mask
census (`mask > 0`, 8-connectivity). **TEST was never read.** The VAL component count
(152) independently reproduces the Phase-12 clean-v3 oracle population.


In [9]:
pop = pd.read_csv(TABLES / 'phase13_clean_v3_training_population_spec.csv')
display(pop)
print('census cache:', p13.POPULATION_CENSUS_CACHE.name,
      '->', p13.POPULATION_CENSUS)


,scope,item,value,unit,source,notes
0,TRAIN / KUB / FULL_FIELD,images,770,images,manifests/phase4_v3/,directory-semantics FOV rule (\Images\ -> FULL_FIELD)
1,TRAIN / KUB / FULL_FIELD,components,1161,connected components,reports/tables/kub_view_type_inventory.csv + mask census,"foreground mask > 0, 8-connectivity"
2,TRAIN / KUB / FULL_FIELD,max_components_per_image,11,components,reports/tables/kub_view_type_inventory.csv + mask census,budget driver for Top-K
3,TRAIN / KUB / FULL_FIELD,images_with_gt_5_components,15,images,reports/tables/kub_view_type_inventory.csv + mask census,motivates a bounded Top-K oracle budget
4,TRAIN / KUB / FULL_FIELD,images_with_zero_components,0,images,reports/tables/kub_view_type_inventory.csv + mask census,FULL_FIELD only
5,VAL / KUB / FULL_FIELD,images,103,images,manifests/phase4_v3/,directory-semantics FOV rule (\Images\ -> FULL_FIELD)
6,VAL / KUB / FULL_FIELD,components,152,connected components,reports/tables/kub_view_type_inventory.csv + mask census,"foreground mask > 0, 8-connectivity"
7,VAL / KUB / FULL_FIELD,max_components_per_image,2,components,reports/tables/kub_view_type_inventory.csv + mask census,budget driver for Top-K
8,VAL / KUB / FULL_FIELD,images_with_gt_5_components,0,images,reports/tables/kub_view_type_inventory.csv + mask census,motivates a bounded Top-K oracle budget
9,VAL / KUB / FULL_FIELD,images_with_zero_components,0,images,reports/tables/kub_view_type_inventory.csv + mask census,FULL_FIELD only


census cache: phase13_population_census_cache.json -> {'TRAIN': {'images': 770, 'components': 1161, 'max_components_per_image': 11, 'images_gt_5_components': 15, 'images_zero_components': 0}, 'VAL': {'images': 103, 'components': 152, 'max_components_per_image': 2, 'images_gt_5_components': 0, 'images_zero_components': 0}}


## 9. Historical vs Clean-v3 Localizer Protocol Diff

Twenty-seven protocol fields compared. Material changes are flagged for supervisor
decision; the scientific core of the localizer (architecture, 128 output, sigma 0.75,
CenterNet focal + soft Dice, AdamW, cosine, the 20-config proposal grid, coverage 0.90)
is carried over unchanged.


In [10]:
diff = pd.read_csv(TABLES / 'phase13_v2_to_clean_v3_localizer_protocol_diff.csv')
display(diff[['field', 'historical_v2', 'proposed_clean_v3', 'same_or_changed',
              'scientifically_material', 'supervisor_decision_required']])
print('SAME:', (diff['same_or_changed'] == 'SAME').sum(),
      '| CHANGED:', (diff['same_or_changed'] == 'CHANGED').sum())
print('material changes:', diff.loc[diff['scientifically_material'], 'field'].tolist())


,field,historical_v2,proposed_clean_v3,same_or_changed,scientifically_material,supervisor_decision_required
0,manifest_version,phase4_v2,phase4_v3,CHANGED,False,False
1,taxonomy,kub_size_v1,kub_size_clean_v3_v1,CHANGED,False,False
2,train_population,FULL_FIELD 771 images / 1162 components,FULL_FIELD 770 images / 1161 components,CHANGED,False,False
3,val_population,FULL_FIELD 106 images / 156 components,FULL_FIELD 103 images / 152 components,CHANGED,False,False
4,val_size_strata,Tiny 73 / Small 29 / Medium 36 / Large 18,Tiny 72 / Small 28 / Medium 36 / Large 16,CHANGED,True,False
5,model_architecture,"KUBLocalizer compact 3-stage U-Net, base_channels 16","KUBLocalizer compact 3-stage U-Net, base_channels 16",SAME,False,False
6,input_channels,1,1,SAME,False,False
7,input_resolution,512,512,SAME,False,False
8,output_heatmap_resolution,"128 (AvgPool2d(4,4) from the stride-4 block)","128 (AvgPool2d(4,4) from the stride-4 block)",SAME,False,False
9,target_sigma,0.75 in 128-space (== 3.0 in 512-space),0.75 in 128-space (== 3.0 in 512-space),SAME,False,False


SAME: 19 | CHANGED: 8
material changes: ['val_size_strata', 'frozen_hard_sets', 'inference_path_defect']


In [11]:
cand = pd.read_csv(TABLES / 'phase13_clean_v3_localizer_protocol_candidate.csv')
display(cand)


,field,historical_v2,proposed_clean_v3,change_reason,scientifically_material,supervisor_decision_required
0,manifest_version,phase4_v2,phase4_v3,clean governance: canonical active-development manifest,False,False
1,taxonomy,kub_size_v1,kub_size_clean_v3_v1,clean governance: frozen clean-v3 taxonomy,False,False
2,train_population,FULL_FIELD 771 images / 1162 components,FULL_FIELD 770 images / 1161 components,clean-v3 population,False,False
3,val_population,FULL_FIELD 106 images / 156 components,FULL_FIELD 103 images / 152 components,clean-v3 population,False,False
4,val_size_strata,Tiny 73 / Small 29 / Medium 36 / Large 18,Tiny 72 / Small 28 / Medium 36 / Large 16,clean-v3 taxonomy recomputation; NOT a relabelling of the v2 strata,True,False
5,model_architecture,"KUBLocalizer compact 3-stage U-Net, base_channels 16","KUBLocalizer compact 3-stage U-Net, base_channels 16",historical architecture is scientifically valid; unchanged,False,False
6,input_channels,1,1,unchanged,False,False
7,input_resolution,512,512,unchanged,False,False
8,output_heatmap_resolution,"128 (AvgPool2d(4,4) from the stride-4 block)","128 (AvgPool2d(4,4) from the stride-4 block)",the Stage-4D bounded redesign is retained; it is the only configuration that...,False,False
9,target_sigma,0.75 in 128-space (== 3.0 in 512-space),0.75 in 128-space (== 3.0 in 512-space),unchanged,False,False


## 10. Proposal Postprocessing Contract

The 14-step postprocessing chain is carried over byte-for-byte. The one trap worth
repeating: `apply_context_margin` returns an **already expanded** `crop_geometry`, so
consumers must not expand it again.


In [12]:
post = pd.read_csv(TABLES / 'phase13_clean_v3_proposal_postprocessing_candidate.csv')
display(post[['step', 'proposed_clean_v3', 'change_reason']])


,step,proposed_clean_v3,change_reason
0,1. threshold,identical rule and identical grid,unchanged
1,2. connected components,identical rule,unchanged
2,3. component scoring,identical rule,unchanged
3,4. component ordering,identical rule,unchanged
4,5. component cap,identical rule,unchanged
5,6. NMS,identical rule,unchanged
6,7. Top-K,identical rule,unchanged
7,8. heatmap->512 mapping,identical rule,unchanged
8,9. 512->native mapping,identical rule,unchanged
9,10. context margin,identical rule,unchanged


## 11. Proposal Metric Contract

Coverage90 is the primary localization hit definition. Two prospective additions are
flagged (`NEW`): proposal precision and unmatched proposals per image — these exist
because the historical Phase-14 failure was an FP-clutter failure that was only diagnosed
post hoc.


In [13]:
mc = pd.read_csv(TABLES / 'phase13_clean_v3_proposal_metric_contract.csv')
display(mc)
print('NEW metrics:', mc.loc[mc['historical_status'].str.startswith('NEW'), 'metric'].tolist())
print('EXCLUDED:', mc.loc[mc['role'].str.startswith('EXCLUDED'), 'metric'].tolist())


,metric,definition,unit,role,historical_status
0,component_coverage,component pixels inside the best proposal ROI / total component pixels,fraction,primary,historical
1,covered90,best coverage >= 0.90,bool,PRIMARY localization hit,historical
2,covered50,best coverage >= 0.50,bool,secondary,historical
3,any_overlap,best coverage > 0,bool,secondary,historical
4,overall_recall90,covered90 components / all GT components,fraction,primary aggregate,historical
5,size_stratum_recall90,covered90 components within TINY/SMALL/MEDIUM/LARGE / stratum total,fraction,primary stratified,historical
6,hard_tiny_recovered90,frozen globally-missed TINY components recovered@90,count,gate criterion A,historical (set must be re-derived clean-v3)
7,global_empty_covered90,frozen global-empty images with >= 1 component covered@90,count,gate criterion B,historical (set must be re-derived clean-v3)
8,multicomponent_recall90,covered90 components within multi-component images / multi-component total,fraction,secondary,historical
9,mean_proposals_per_image,mean proposal count per image,count,efficiency / gate criterion F,historical


NEW metrics: ['proposal_precision', 'unmatched_proposals_per_image']
EXCLUDED: ['localizer_dice']


## 12. Downstream Integration Candidate

The Phase-13 -> Phase-14 handoff, with the distribution-mismatch fix promoted to an
explicit material change.


In [14]:
ds = pd.read_csv(TABLES / 'phase13_clean_v3_downstream_integration_candidate.csv')
display(ds[['item', 'proposed_clean_v3', 'scientifically_material', 'supervisor_decision_required']])


,item,proposed_clean_v3,scientifically_material,supervisor_decision_required
0,role of Phase 13,"Produce GT-free proposals consumed by a regional refiner, evaluated as a thr...",False,False
1,refiner input geometry,identical rule,False,False
2,fusion rule,identical rule,False,False
3,final threshold,0.50,False,False
4,zero-proposal fallback,identical rule,False,False
5,refiner training crops,PROPOSE: draw training crops from Stage-4D predicted proposals (with an expl...,True,True
6,refiner training population,clean-v3 FULL_FIELD (770 train images / 1161 components),False,False
7,oracle budget,TopK=5 largest GT components (clean-v3 TRAIN max is still 11),False,False
8,FP-burden reporting,"report proposal precision, unmatched proposals/image and the FP burden prosp...",True,False
9,cross-phase gate,Phases 14-18 remain PENDING_POST_PHASE13_REGIONAL_INTEGRATION_GATE; no autom...,False,False


## 13. Oracle-Retention Metric Contract (dimension B arithmetic)

`T_g` (global coverage90), `T_o` (oracle coverage90), `T_p` (predicted coverage90),
`G = T_o - T_g`, `R = (T_p - T_g) / G`. The historical v2 values are shown for context
only. **None of the clean-v3 values exist yet** — Phase 13 exists to measure them. If
`G <= 0`, retention is undefined and the gate must report `NO_GO` / `REVISE` rather than
a ratio.


In [15]:
ret = pd.read_csv(TABLES / 'phase13_oracle_retention_metric_contract.csv')
display(ret[['symbol', 'name', 'status', 'notes']])
print()
print('historical v2 retention:', {k: p13.PHASE14_GATE[k] for k in ('T_g', 'T_o', 'T_p', 'G', 'R')})
print('clean-v3 retention: NOT_YET_COMPUTED (population 103/152)')


,symbol,name,status,notes
0,T_g,GLOBAL baseline coverage90,NOT_YET_COMPUTED,replaces the historical v2 value 0.780822
1,T_o,ORACLE REGION coverage90,NOT_YET_COMPUTED,the clean-v3 upper bound; must NOT be copied from the v2 value 1.000000
2,T_p,PREDICTED REGION coverage90,NOT_YET_COMPUTED,the quantity Phase 13 exists to measure
3,G,Refiner potential,NOT_YET_COMPUTED,denominator of the retention ratio; if G <= 0 retention is undefined and the...
4,R,Proposal retention fraction,NOT_YET_COMPUTED,"historical v2 value was exactly 0.500000; the clean-v3 value is a CANDIDATE,..."
5,D_g,GLOBAL full-field Dice,AVAILABLE_REFERENCE,"per-model: {""UNet"": 0.59418, ""Attention U-Net"": 0.629805, ""UNet++"": 0.640516..."
6,D_o,ORACLE REGION full-field Dice,AVAILABLE_REFERENCE,"per-model: {""UNet"": 0.724722, ""Attention U-Net"": 0.699595, ""UNet++"": 0.72667..."
7,D_p,PREDICTED REGION full-field Dice,NOT_YET_COMPUTED,the historical v2 value was 0.383905 and drove the Phase-14 Step-3 failure
8,object_recall_global_vs_oracle,clean-v3 object recall GLOBAL -> ORACLE,AVAILABLE_REFERENCE,"per-model (global, oracle): {""UNet"": [0.7864, 0.9951], ""Attention U-Net"": [0..."
9,retention_scope,architecture scope,CANDIDATE,no reference model is selected; heterogeneity must be visible



historical v2 retention: {'T_g': 0.780822, 'T_o': 1.0, 'T_p': 0.890411, 'G': 0.219178, 'R': 0.5}
clean-v3 retention: NOT_YET_COMPUTED (population 103/152)


## 14. Gate Dimension B Candidate + C/D Bridge

Dimension B (`PRACTICAL_LOCALIZATION_RETENTION`) is the only gate dimension Phase 13 can
move. Dimensions C and D are re-examined with Phase-13 evidence; dimension A is already
`SUPPORTED_DESCRIPTIVE_EVIDENCE` from Phase 12. **No numeric threshold is frozen.**


In [16]:
gb = pd.read_csv(TABLES / 'phase13_gate_dimension_b_candidate.csv')
display(gb)

cd = pd.read_csv(TABLES / 'phase13_gate_dimensions_cd_bridge.csv')
display(cd)


,dimension_id,dimension,question,evidence_fields,criterion_status,candidate_criterion,numeric_threshold
0,B,PRACTICAL_LOCALIZATION_RETENTION,"In Phase 13, how much of the clean-v3 ORACLE regional-resolution benefit is ...","T_g, T_o, T_p, G, R, D_g, D_o, D_p, size-stratum recall90, proposal precisio...",CANDIDATE_FOR_SUPERVISOR_FREEZE,Candidate: R clearly above zero AND D_p not materially below D_g AND the rec...,NOT_FROZEN


,dimension_id,dimension,status_after_phase12,phase13_evidence_required,bridge_to,candidate_criterion
0,A,ORACLE_MECHANISM_VALUE,SUPPORTED_DESCRIPTIVE_EVIDENCE,No further Phase-13 evidence required; Phase-13 evidence must not contradict...,Phase 12 -> Phase 13,Positive median paired delta Dice across all four clean-v3 models (descripti...
1,B,PRACTICAL_LOCALIZATION_RETENTION,NOT_EVALUABLE_UNTIL_PHASE13,Phase-13 T_g / T_o / T_p / G / R on the clean-v3 population; re-derived clea...,Phase 13 -> REGIONAL_INTEGRATION_GATE_AFTER_PHASE13,Candidate: R above zero; exact fraction NOT frozen.
2,C,GLOBAL_PRESERVATION_FP_COST,MIXED_DESCRIPTIVE_EVIDENCE,"Phase-13 proposal precision, unmatched proposals/image, FP component burden,...",Phase 13 -> Phase 14 (conditional),Candidate: no material global degradation (D_p drop bounded) with a prospect...
3,D,GENERALITY,SUPPORTED_WITH_HETEROGENEITY,Phase-13 per-architecture retention spread and per-stratum retention.,Phase 13 -> RA-MedFiLM regional-mechanism reuse,Candidate: retention not confined to one architecture or one size stratum.


## 15. Future Visualization Protocol

Declared **before** results exist, so the Phase-13 evidence package cannot be
retro-fitted to a convenient visual.


In [17]:
viz = pd.read_csv(TABLES / 'phase13_future_visualization_protocol.csv')
display(viz)


,id,visualization,form,basis
0,A,GLOBAL vs ORACLE vs PREDICTED coverage90,grouped bar,clean-v3 FULL_FIELD 103 cases / 152 components
1,B,retention fraction R per architecture,bar with the G denominator annotated,per model; no reference model
2,C,size-stratified recall90 across the three conditions,grouped bar TINY/SMALL/MEDIUM/LARGE,clean-v3 taxonomy strata
3,D,hard-set recovery (re-derived clean-v3),bar recovered/not-recovered,"clean-v3 hard sets, NOT the v2 16/7"
4,E,proposal count vs coverage trade-off,scatter with the feasibility region shaded,proposals/image vs recall90
5,F,ROI area fraction vs coverage,scatter with the area budget line,union area fraction
6,G,heatmap + proposal overlays,"multipage panel (image, heatmap, proposals)",deterministic selection; >= 12 cases
7,H,TP/FP/FN error maps for the predicted-region condition,multipage panel,>= 16 deterministic cases
8,I,Dice distribution GLOBAL vs ORACLE vs PREDICTED,paired violin/box,paired within case and architecture
9,J,proposal precision vs coverage,scatter,prospective FP-cost reporting


## 16. Artifact Registry

Content-addressed SHA-256 for every Part-1A artifact.


In [18]:
reg = pd.DataFrame([{'artifact': k, 'sha256': v} for k, v in sorted(hashes.items())])
display(reg)


,artifact,sha256
0,phase13_clean_v3_downstream_integration_candidate.csv,1314726c25086b701a8fc0ca6d0a5a9767b7925534796d554008cad4900a36b6
1,phase13_clean_v3_localizer_protocol_candidate.csv,fa78872a5b011237b3a1747b7a912bca6e0b2db71f81d8a8b89c7df73a045b71
2,phase13_clean_v3_localizer_protocol_candidate.json,bf4c2db8be89d0ca03a13e82b60e830c786745dfa85befaf09eee25fb52c5403
3,phase13_clean_v3_localizer_protocol_candidate.md,039069cf095305275bcb57e4ee9dd1247790d23854ef10f896fcc28c168baef6
4,phase13_clean_v3_proposal_metric_contract.csv,47a737e3c4b0a0d550efcdb4f72f8508b9406c3aa34ae8dd8dd338a5cf91b09c
5,phase13_clean_v3_proposal_postprocessing_candidate.csv,38e0b1204c50d9f6391c79566fa3052a8a1c87d01dc2d5c412d8bcb5ee9938d6
6,phase13_clean_v3_training_population_spec.csv,53e9f2cef5ad73f306e57478d6ce37a8835f5f685aa7a8c9416b540836aaa7e1
7,phase13_future_visualization_protocol.csv,154e58ce196a624de6f6c97b71bfd5aa42225e2303746089a0d72da3f3f5ea2d
8,phase13_gate_dimension_b_candidate.csv,63c55c10c0f6e0d26ee47d58b17e7c913a1de1a54ed017a473896f67ef25a159
9,phase13_gate_dimensions_cd_bridge.csv,6faf39de4a755e368a53d9c4ed4a29c8456ce0c78f43251f9d3103e0b72c7f33


## 17. Governance Summary and STOP

- TEST content reads **0**; TEST metrics **NONE**; external data **NONE**.
- Localizer training **0**; localizer inference **0**; proposals generated **0**.
- No checkpoint / history / config / manifest was modified.
- Phase 14 **not executed**; Phases 15–18 **not started**.
- No winner declared; no reference architecture selected.
- Regional mechanism **not promoted**; `REGIONAL_INTEGRATION_GATE_AFTER_PHASE13` **NOT_DECIDED**.
- Final architecture `RA-MedFiLM-Seg++` remains **`PROPOSED_NOT_IMPLEMENTED`**.

**STOP after Phase 13 Part 1A — awaiting supervisor review of the protocol candidate.**


In [19]:
print('PHASE 13 PART 1A — COMPLETE (protocol/definition only)')
print('test_reads=0  training=0  inference=0  proposals=0  phase14_executed=False')
print('gate=NOT_DECIDED  winner=none  reference_model=none  regional_mechanism_promoted=False')


PHASE 13 PART 1A — COMPLETE (protocol/definition only)
test_reads=0  training=0  inference=0  proposals=0  phase14_executed=False
gate=NOT_DECIDED  winner=none  reference_model=none  regional_mechanism_promoted=False
